# 08 · Export the chosen model for deployment (Streamlit)

Notebook 05 chose the segmentation (k-prototypes, k = 6, seg-v1.0) and saved the labels, but not the fitted model.
This notebook **re-fits the same final model** (same features, same random seed, same sample, so the result is identical),
checks that it reproduces `customer_segments_seg-v1.0.csv` exactly, and writes everything the Streamlit app needs to `app/model/`:

| File | What it is |
|---|---|
| `kprototypes_seg-v1.0.joblib` | the fitted k-prototypes model |
| `model_meta.json` | feature lists, the scaler (mean and scale), cluster → segment map, allowed categories, metrics |
| `segment_profiles.csv` | one row per segment: size, value, conversion, late payment, prototype values, tier, desk, action |
| `model_comparison.csv` | every model tried: k-means baseline and k-prototypes for k = 2 to 10 |
| `customer_lookup.csv` | customer_id → segment (for the look-up page) |
| `example_batch.csv` | 200 customers in the raw input format, to test batch scoring |

The app then runs **without the database**: it only reads these files. Run time: about 2 minutes.

In [1]:
# First time only:  %pip install kmodes scikit-learn joblib sqlalchemy psycopg2-binary

In [2]:
import json
import joblib
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from kmodes.kprototypes import KPrototypes
from IPython.display import display, Markdown
import vx

vx.connect()
APP = vx.ROOT / "app"
MD = APP / "model"
MD.mkdir(parents=True, exist_ok=True)
say = lambda s: display(Markdown(s))

# The settings notebook 05 used for the final model: do not change them, or the labels will differ
SEED, N_FINAL, GAMMA, K, VERSION = 42, 20000, 1.0, 6, "seg-v1.0"

Connected to database 'Veltrix' on localhost:5432 as postgres


## 1. Re-fit the final model exactly as in notebook 05

In [3]:
spec = json.load(open(vx.REPORTS / "04_features" / "feature_spec.json"))
NUM_RAW, CAT, MONEY = spec["numeric"], spec["categorical"], spec["money_log1p"]
NUM = ["z_" + c for c in NUM_RAW]
m = vx.Q("SELECT * FROM feat.customer_features_model ORDER BY customer_id")
X = pd.concat([m[NUM].astype(float), m[CAT].astype(str)], axis=1)
CI = list(range(len(NUM), len(NUM) + len(CAT)))
XA = X.to_numpy(dtype=object)
final_idx = X.sample(N_FINAL, random_state=SEED + 2).index
model = KPrototypes(n_clusters=K, init="Cao", n_init=5, random_state=SEED, gamma=GAMMA, max_iter=50)
model.fit(XA[X.index.get_indexer(final_idx)], categorical=CI)
labels = model.predict(XA, categorical=CI)

saved = pd.read_csv(vx.REPORTS / "05_model" / f"customer_segments_{VERSION}.csv")
chk = pd.DataFrame({"customer_id": m.customer_id, "cluster": labels}).merge(saved, on="customer_id")
cmap = chk.groupby("cluster").segment_id.agg(lambda s: s.mode()[0])
agree = (chk.cluster.map(cmap) == chk.segment_id).mean()
assert agree > 0.999, f"Re-fit does not reproduce the saved labels ({agree:.2%}): check the settings above"
names = saved.drop_duplicates("segment_id").set_index("segment_id").segment_name
cluster_to_segment = {int(c): {"segment_id": int(s), "segment_name": names[s]} for c, s in cmap.items()}
say(f"Re-fitted model reproduces the saved seg-v1.0 labels for **{agree:.2%}** of {len(chk):,} customers.")
pd.DataFrame(cluster_to_segment).T

Re-fitted model reproduces the saved seg-v1.0 labels for **100.00%** of 100,573 customers.

,segment_id,segment_name
0,2,Home movers
1,3,Renters
2,1,First-time buyers
3,4,Buy-to-let investors
4,5,Overseas investors
5,6,Window shoppers


## 2. Segment profiles (from the features and the held-out outcomes)

In [4]:
oc = vx.Q("""SELECT customer_id, n_closed_leads, n_won_leads, deal_value_gbp, payments_due, payments_late_or_overdue
             FROM feat.customer_outcomes""")
d = m.merge(saved, on="customer_id").merge(oc, on="customer_id", how="left").fillna({"n_closed_leads": 0, "n_won_leads": 0,
                                                                                    "deal_value_gbp": 0, "payments_due": 0, "payments_late_or_overdue": 0})
g = d.groupby(["segment_id", "segment_name"])
P = g.agg(customers=("customer_id", "size"), closed=("n_closed_leads", "sum"), won=("n_won_leads", "sum"),
          value=("deal_value_gbp", "sum"), due=("payments_due", "sum"), late=("payments_late_or_overdue", "sum"),
          median_budget_gbp=("budget_gbp_imputed", "median"), median_rent_pcm_gbp=("rent_budget_pcm_imputed", "median"),
          median_sessions=("sessions_total", "median"), night_share=("night_session_share", "mean"),
          virtual_tour_share=("virtual_tour_share", "mean"), mobile_share=("mobile_share", "mean"),
          pct_overseas=("is_overseas", "mean"), pct_no_enquiry=("enquiries_total", lambda s: (s == 0).mean()),
          top_region=("preferred_region", lambda s: s.mode()[0]), top_property_type=("preferred_property_type", lambda s: s.mode()[0]),
          top_channel=("lead_source_primary", lambda s: s[s != "No lead"].mode()[0] if (s != "No lead").any() else "No lead"),
          top_finance=("finance_type", lambda s: s.mode()[0]), top_buyer_position=("buyer_position", lambda s: s.mode()[0])).reset_index()
P["pct_customers"] = 100 * P.customers / P.customers.sum()
P["pct_deal_value"] = 100 * P.value / P.value.sum()
P["conversion_pct"] = 100 * P.won / P.closed.replace(0, np.nan)
P["late_payment_pct"] = 100 * P.late / P.due.replace(0, np.nan)
for c in ["night_share", "virtual_tour_share", "mobile_share", "pct_overseas", "pct_no_enquiry"]:
    P[c] = 100 * P[c]
# Priority tier: value of an average closed lead vs all leads (same rule as week3/11 STEP 1)
P["value_per_lead_index"] = 100 * (P.value / P.closed.replace(0, np.nan)) / (P.value.sum() / P.closed.sum())
P["priority_tier"] = np.select([P.value_per_lead_index >= 115, P.value_per_lead_index >= 60], ["P1", "P2"], "P3")
# Desk, agent level and action: from the database if Week 3 ran on seg-v1.0, else from the tier rules
if vx.exists("clean.segment_actions") and vx.segment_model() == VERSION:
    act = vx.Q("SELECT segment_name, routed_desk, agent_level, reply_target, best_first_contact, recommended_action FROM clean.segment_actions")
    P = P.merge(act, on="segment_name", how="left")
else:
    say("Week 3 views are not on seg-v1.0 yet: desk and action text come from the tier rules.")
    RULE = {"P1": ("Senior agent first", "First reply within 1 hour, 7 days a week"),
            "P2": ("Mid or senior agent", "First reply within 4 hours (working hours)"),
            "P3": ("Any agent or automated nurture", "Same day; automated reply immediately")}
    P["routed_desk"] = np.where(P.pct_overseas > 50, "International desk", np.where(P.segment_name.str.contains("investor", case=False), "Investor desk",
                       np.where(P.segment_name.str.contains("Renter"), "Regional lettings desks", "Regional sales desks")))
    P["agent_level"] = P.priority_tier.map(lambda t: RULE[t][0])
    P["reply_target"] = P.priority_tier.map(lambda t: RULE[t][1])
    P["best_first_contact"] = "Phone"
    P["recommended_action"] = P.priority_tier + ": route to " + P.routed_desk + " (" + P.agent_level + "); " + P.reply_target + "."
P = P.drop(columns=["closed", "won", "value", "due", "late"]).round(2).sort_values("segment_id")
P.to_csv(MD / "segment_profiles.csv", index=False)
P[["segment_id", "segment_name", "customers", "pct_customers", "pct_deal_value", "conversion_pct", "priority_tier", "routed_desk"]]

Week 3 views are not on seg-v1.0 yet: desk and action text come from the tier rules.

,segment_id,segment_name,customers,pct_customers,pct_deal_value,conversion_pct,priority_tier,routed_desk
0,1,First-time buyers,27206,27.05,21.53,12.58,P2,Regional sales desks
1,2,Home movers,23919,23.78,20.99,17.06,P1,Regional sales desks
2,3,Renters,20263,20.15,2.78,27.29,P3,Regional lettings desks
3,4,Buy-to-let investors,15563,15.47,43.99,19.19,P1,Investor desk
4,5,Overseas investors,7418,7.38,9.90,14.09,P1,International desk
5,6,Window shoppers,6204,6.17,0.81,7.13,P3,Regional sales desks


## 3. Model comparison (every model tried)

In [5]:
sel = pd.read_csv(vx.REPORTS / "05_model" / "model_selection.csv")
ev = X.sample(3000, random_state=SEED + 1).index
km = []
for k in range(2, 11):
    kk = KMeans(n_clusters=k, n_init=4, random_state=SEED).fit(X[NUM])
    km.append(dict(model="k-means (numeric only, baseline)", k=k, silhouette=silhouette_score(X.loc[ev, NUM], kk.predict(X.loc[ev, NUM]))))
comp = pd.concat([pd.DataFrame(km),
                  sel.assign(model="k-prototypes (mixed, Gower silhouette)").rename(columns={"silhouette_gower": "silhouette"})
                  [["model", "k", "silhouette", "smallest_segment_pct", "ari_kprototypes_vs_hierarchical", "stability_mean_ari"]]])
comp["chosen"] = (comp.model.str.startswith("k-prototypes")) & (comp.k == K)
comp.round(3).to_csv(MD / "model_comparison.csv", index=False)
comp.round(3)

,model,k,silhouette,smallest_segment_pct,ari_kprototypes_vs_hierarchical,stability_mean_ari,chosen
0,"k-means (numeric only, baseline)",2,0.319,NaN,NaN,NaN,False
1,"k-means (numeric only, baseline)",3,0.358,NaN,NaN,NaN,False
2,"k-means (numeric only, baseline)",4,0.200,NaN,NaN,NaN,False
3,"k-means (numeric only, baseline)",5,0.198,NaN,NaN,NaN,False
4,"k-means (numeric only, baseline)",6,0.199,NaN,NaN,NaN,False
5,"k-means (numeric only, baseline)",7,0.193,NaN,NaN,NaN,False
6,"k-means (numeric only, baseline)",8,0.204,NaN,NaN,NaN,False
7,"k-means (numeric only, baseline)",9,0.215,NaN,NaN,NaN,False
8,"k-means (numeric only, baseline)",10,0.221,NaN,NaN,NaN,False
0,"k-prototypes (mixed, Gower silhouette)",2,0.279,21.012,NaN,NaN,False


## 4. Save the model and its metadata

In [6]:
joblib.dump(model, MD / f"kprototypes_{VERSION}.joblib", compress=3)
crow = sel.set_index("k").loc[K]
meta = {
    "model_version": VERSION, "algorithm": "k-prototypes (kmodes)", "k": K, "gamma": GAMMA, "init": "Cao", "random_state": SEED,
    "trained_on": f"{N_FINAL:,} customer sample; all {len(m):,} customers assigned", "data_to": "2026-09-30",
    "numeric": NUM_RAW, "categorical": CAT, "money_log1p": MONEY,
    "scaler_mean": spec["scaler_mean"], "scaler_scale": spec["scaler_scale"],
    "categories": {c: sorted(m[c].astype(str).unique().tolist()) for c in CAT},
    "numeric_ranges": {c: [float(m[c].quantile(0.01)), float(m[c].median()), float(m[c].quantile(0.99))]
                       for c in NUM_RAW if c in m.columns},
    "budget_ranges": {"budget_gbp": [float(m.budget_gbp_imputed.quantile(q)) for q in (0.01, 0.5, 0.99)],
                      "rent_budget_pcm_gbp": [float(m.rent_budget_pcm_imputed.quantile(q)) for q in (0.01, 0.5, 0.99)]},
    "cluster_to_segment": cluster_to_segment,
    "metrics": {"silhouette_gower": float(crow.silhouette_gower), "bootstrap_ari_mean": float(crow.stability_mean_ari),
                "smallest_segment_pct": float(crow.smallest_segment_pct), "reproduces_saved_labels": float(agree)},
}
json.dump(meta, open(MD / "model_meta.json", "w"), indent=2)
saved.to_csv(MD / "customer_lookup.csv", index=False)

# Example batch file: raw inputs (what a new customer record looks like), 200 customers
raw = m.sample(200, random_state=7)
ex = pd.DataFrame({"customer_id": raw.customer_id, "budget_gbp": raw.budget_gbp_imputed, "rent_budget_pcm_gbp": raw.rent_budget_pcm_imputed})
for c in NUM_RAW:
    if c not in MONEY:
        ex[c] = raw[c].values
for c in CAT:
    ex[c] = raw[c].astype(str).values
ex.to_csv(MD / "example_batch.csv", index=False)
say(f"Saved to `app/model/`: " + ", ".join(sorted(p.name for p in MD.iterdir())))

Saved to `app/model/`: customer_lookup.csv, example_batch.csv, kprototypes_seg-v1.0.joblib, model_comparison.csv, model_meta.json, segment_profiles.csv

## 5. Check: score the example batch from raw inputs, as the app does

In [7]:
def transform(df):
    Z = pd.DataFrame(index=df.index)
    for c in NUM_RAW:
        x = df[c].astype(float)
        x = np.log1p(x.fillna(0)) if c in MONEY else x.fillna(meta["numeric_ranges"][c][1])
        Z[c] = (x - meta["scaler_mean"][c]) / meta["scaler_scale"][c]
    for c in CAT:
        Z[c] = df[c].astype(str)
    return Z.to_numpy(dtype=object)

pred = model.predict(transform(ex), categorical=CI)
ok = (pd.Series(pred).map(lambda c: cluster_to_segment[int(c)]["segment_id"]).values ==
      ex[["customer_id"]].merge(saved, on="customer_id").segment_id.values).mean()
say(f"Scoring from raw inputs matches the saved segment for **{ok:.1%}** of the 200 example customers.")

Scoring from raw inputs matches the saved segment for **100.0%** of the 200 example customers.